# Fix: LODO Permutation Test — Corrected Tail Direction

## What this notebook fixes

In the original NB04, the LODO (Leave-One-Dataset-Out) permutation p-value was computed as:
```python
perm_p = float(np.mean(np.array(perm_rs) >= rho))   # WRONG: right-tailed
```

**Why this is wrong:** The analysis correlates meta-analysis **rank position** (0 = best gene,  
sorted by descending π-value) with **absolute log2FC** in the held-out dataset.  
A correctly predictive meta-analysis produces **negative Spearman ρ** because:
- Rank 0 (lowest index) = highest π-value gene → should have the **highest** |FC| in held-out  
- So high |FC| maps to low rank index → **negative ρ is the correct signal**

To test whether this negative ρ is more extreme than chance, we need a **left-tailed test**:
```python
perm_p = float(np.mean(np.array(perm_rs) <= rho))   # CORRECT: left-tailed
```

The original code was testing whether ρ was unusually *positive* — which it never was,  
giving perm_p ≈ 1.0 for all 8 folds and making all folds appear non-significant.

## What this notebook does
1. Loads all necessary data (same as NB04)
2. Re-runs the LODO analysis with the corrected left-tailed permutation test
3. Also reads the fixed `common_genes.csv` (run `Fix_NB03_CrossSpecies_DirectionConcordant.ipynb` first)
4. Saves corrected `lodo_rank_correlation.csv` and `validation_summary.json`
5. Regenerates `NB5_FigB_LODO_RankCorrelation` with correct significance annotations
6. Regenerates `NB5_FigSummary_All_Validation`

## Prerequisites
- Run `Fix_NB03_CrossSpecies_DirectionConcordant.ipynb` first
- Requires the same data files as NB04 (`Results/clf_human_LN_final.csv`, `Input Data/H_L_N/` etc.)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scipy statsmodels matplotlib seaborn

import pandas as pd
import numpy as np
import json, os, warnings
warnings.filterwarnings('ignore')

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr, mannwhitneyu, binomtest

matplotlib.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.labelsize'   : 12,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : 'white',
    'savefig.dpi'      : 300,
})

DPI = 300
CLR = {'Known':'#2166AC', 'Novel':'#D6604D', 'Human':'#1A78C2',
       'Up':'#D73027', 'Down':'#4575B4'}

# ── Paths ──────────────────────────────────────────────────────────────────
BASE     = '/content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper'
IN_DIR   = os.path.join(BASE, 'Input Data')
RES      = os.path.join(BASE, 'Results')
NB02_TAB = os.path.join(RES, 'NB02', 'Tables')
NB04     = os.path.join(RES, 'NB04')
NB04_FIG = os.path.join(NB04, 'Figures')
NB04_TAB = os.path.join(NB04, 'Tables')
NB04_SUP = os.path.join(NB04, 'Supplementary_Files')

for d in [NB04_FIG, NB04_TAB, NB04_SUP]:
    os.makedirs(d, exist_ok=True)

def save_fig(fig, name):
    for ext in ['png', 'jpg', 'pdf']:
        path = os.path.join(NB04_FIG, f'{name}.{ext}')
        fig.savefig(path, dpi=DPI if ext != 'pdf' else None,
                    bbox_inches='tight', facecolor='white',
                    format=ext if ext != 'jpg' else 'jpeg')
    print(f'  Saved: {name}.png / .jpg / .pdf')
    plt.show()
    plt.close(fig)

print('Setup complete.')

Setup complete.


In [3]:
# ── Load meta-analysis results and per-dataset DEGs ────────────────────────
# Load the classified H_L/N results (same file NB04 uses)
clf_path = os.path.join(NB02_TAB, 'clf_human_LN_final.csv')
if not os.path.exists(clf_path):
    clf_path = os.path.join(RES, 'clf_human_LN_final.csv')  # fallback

sig_human = pd.read_csv(clf_path)
sig_human = sig_human[sig_human['meta_significant'] == True].copy()
print(f'Meta-significant H_L/N genes: {len(sig_human)}')

# Status dict for direction concordance later
status_h = dict(zip(sig_human['Gene_Symbol'], sig_human['Status']))

# Load individual dataset DEGs for H_L/N
h_ln_dir = os.path.join(IN_DIR, 'H_L_N')
print(f'\nLooking for per-dataset DEGs in: {h_ln_dir}')

human_ds = {}
if os.path.isdir(h_ln_dir):
    for fname in os.listdir(h_ln_dir):
        if fname.endswith('.csv') or fname.endswith('.txt'):
            try:
                df = pd.read_csv(os.path.join(h_ln_dir, fname))
                # Standardise column names (case-insensitive)
                df.columns = [c.strip() for c in df.columns]
                col_map = {c: c for c in df.columns}
                for c in df.columns:
                    if c.lower() in ('gene_symbol', 'gene', 'genesymbol'):
                        col_map[c] = 'Gene_Symbol'
                    elif c.lower() in ('log2fc', 'log2_fc', 'logfc', 'log2foldchange'):
                        col_map[c] = 'log2FC'
                df = df.rename(columns=col_map)
                if 'Gene_Symbol' in df.columns and 'log2FC' in df.columns:
                    key = fname.replace('.csv','').replace('.txt','')
                    human_ds[key] = df[['Gene_Symbol','log2FC']].dropna()
                    print(f'  Loaded: {key}  ({len(df)} genes)')
            except Exception as e:
                print(f'  Skipped {fname}: {e}')

print(f'\nTotal datasets loaded: {len(human_ds)}')

Meta-significant H_L/N genes: 176

Looking for per-dataset DEGs in: /content/drive/MyDrive/Research/Atopic Dermatitis/Research Paper/Input Data/H_L_N

Total datasets loaded: 0


In [4]:
# ── Helper: run meta-analysis on a subset of datasets ─────────────────────
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests

def run_meta_subset(ds_dict, gene_list, min_ds=1):
    """
    Fisher's combined probability on a subset of datasets.
    Returns DataFrame with Gene_Symbol, median_fc, padj, n_datasets.
    """
    rows = []
    for gene in gene_list:
        ps, fcs = [], []
        for df in ds_dict.values():
            match = df[df['Gene_Symbol'] == gene]
            if match.empty:
                continue
            row = match.iloc[0]
            # Need a p-value column; try common names
            for pcol in ['p_value','pvalue','P.Value','PValue','adj_pval']:
                if pcol in row.index:
                    try:
                        ps.append(max(float(row[pcol]), 1e-300))
                    except:
                        pass
                    break
            try:
                fcs.append(float(row['log2FC']))
            except:
                pass
        if len(ps) < min_ds or len(fcs) < min_ds:
            continue
        if len(ps) == 1:
            combined_p = ps[0]
        else:
            _, combined_p = combine_pvalues(ps, method='fisher')
        rows.append({'Gene_Symbol': gene, 'median_fc': np.median(fcs),
                     'raw_p': combined_p, 'n_datasets': len(ps)})
    if not rows:
        return pd.DataFrame()
    res = pd.DataFrame(rows)
    res['padj'] = multipletests(res['raw_p'], method='fdr_bh')[1]
    return res

In [5]:
# ── LODO with CORRECTED left-tailed permutation test ──────────────────────
N_PERM  = 1000
lodo_rows = []

if human_ds and len(human_ds) >= 3:
    all_sig = sig_human['Gene_Symbol'].tolist()
    print(f'LODO: {len(human_ds)} folds, {len(all_sig)} genes, {N_PERM} permutations each...')
    print()

    for held_out in human_ds:
        train   = {k: v for k, v in human_ds.items() if k != held_out}
        held_df = human_ds[held_out]

        meta_tr = run_meta_subset(train, all_sig, min_ds=1)
        if meta_tr.empty:
            print(f'  {held_out}: skipped (no meta result)')
            continue

        # π-value proxy: |median_fc| * -log10(padj)
        meta_tr['pi_tr'] = (
            meta_tr['median_fc'].abs() *
            (-np.log10(meta_tr['padj'].clip(lower=1e-300)))
        )
        # Sort by descending π: rank 0 = best gene
        meta_tr = meta_tr.sort_values('pi_tr', ascending=False).reset_index(drop=True)

        # Held-out absolute FC
        held_fc = dict(zip(held_df['Gene_Symbol'], held_df['log2FC'].abs()))
        common  = [g for g in meta_tr['Gene_Symbol'] if g in held_fc]
        if len(common) < 10:
            print(f'  {held_out}: skipped (only {len(common)} common genes)')
            continue

        # Rank position (0 = highest π)
        ranks = [meta_tr[meta_tr['Gene_Symbol'] == g].index[0] for g in common]
        fcs   = [held_fc[g] for g in common]

        rho, pval = spearmanr(ranks, fcs)

        # ── CORRECTED: left-tailed permutation ────────────────────────────
        # Expected: lower rank (better gene) → higher |FC| → NEGATIVE ρ
        # Significance test: P(perm_ρ ≤ observed_ρ)  [left tail]
        rng_l    = np.random.default_rng(42)
        perm_rs  = [spearmanr(rng_l.permutation(ranks), fcs)[0] for _ in range(N_PERM)]
        perm_p   = float(np.mean(np.array(perm_rs) <= rho))   # LEFT-TAILED (CORRECTED)

        lodo_rows.append({
            'HeldOut'     : held_out,
            'N_genes'     : len(common),
            'Spearman_rho': round(rho, 4),
            'p_value'     : round(pval, 4),
            'perm_p'      : round(perm_p, 4),
            'Significant' : perm_p < 0.05,
        })
        sig_label = '✓ significant' if perm_p < 0.05 else 'ns'
        print(f'  {held_out:<28}: ρ={rho:+.3f}  perm_p={perm_p:.4f}  n={len(common)}  [{sig_label}]')

    lodo_corr_df = pd.DataFrame(lodo_rows)
    print(f'\nMean ρ          : {lodo_corr_df["Spearman_rho"].mean():.3f}')
    print(f'Significant folds: {lodo_corr_df["Significant"].sum()} / {len(lodo_corr_df)}')

    # Save corrected CSV
    lodo_corr_df.to_csv(os.path.join(RES, 'lodo_rank_correlation.csv'), index=False)
    lodo_corr_df.to_csv(os.path.join(NB04_TAB, 'lodo_rank_correlation.csv'), index=False)
    print('\nSaved lodo_rank_correlation.csv')

else:
    print('⚠ Need ≥ 3 datasets for LODO.')
    lodo_corr_df = pd.DataFrame()

⚠ Need ≥ 3 datasets for LODO.


In [6]:
# ── Regenerate NB5_FigB: LODO bar chart ────────────────────────────────────
if not lodo_corr_df.empty:
    fig, ax = plt.subplots(figsize=(max(7, len(lodo_corr_df) * 1.1), 5.5))

    bclrs = [CLR['Human'] if r['Significant'] else '#AAAAAA'
             for _, r in lodo_corr_df.iterrows()]
    bars  = ax.bar(range(len(lodo_corr_df)), lodo_corr_df['Spearman_rho'],
                   color=bclrs, edgecolor='white', width=0.6)

    for b, (_, r) in zip(bars, lodo_corr_df.iterrows()):
        sym = '*' if r['Significant'] else 'ns'
        ypos = b.get_height() + 0.005 if b.get_height() >= 0 else b.get_height() - 0.03
        ax.text(b.get_x() + b.get_width() / 2, ypos,
                f'{r["Spearman_rho"]:.2f}\n{sym}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.axhline(0, color='#888', lw=0.8, ls='--')
    mean_rho = lodo_corr_df['Spearman_rho'].mean()
    ax.axhline(mean_rho, color='#333', lw=1.5, ls=':')
    ax.text(len(lodo_corr_df) - 0.4, mean_rho + 0.008,
            f'Mean ρ = {mean_rho:.3f}', fontsize=9, color='#333')

    ax.set_xticks(range(len(lodo_corr_df)))
    ax.set_xticklabels(lodo_corr_df['HeldOut'], rotation=25, ha='right', fontsize=10)
    ax.set_ylabel('Spearman ρ  (meta-rank vs held-out |log₂FC|)\n'
                  '(negative ρ = correct direction: lower rank → higher FC)')
    ax.set_title('Leave-One-Dataset-Out Rank Correlation — CORRECTED\n'
                 '(* left-tailed perm p < 0.05;  negative ρ = valid signature)',
                 fontweight='bold')

    patches = [
        mpatches.Patch(color=CLR['Human'], label='Significant (perm p < 0.05)'),
        mpatches.Patch(color='#AAAAAA',   label='Not significant'),
    ]
    ax.legend(handles=patches, frameon=False, fontsize=10)
    fig.tight_layout()
    save_fig(fig, 'NB5_FigB_LODO_RankCorrelation')

In [7]:
# ── Read corrected binomial result (from Fix_NB03 notebook) ───────────────
binom_json_path = os.path.join(RES, 'cross_species_binomial.json')

binom_stats = {'n_concordant': 0, 'n_total': 0, 'p_value': 1.0}
if os.path.exists(binom_json_path):
    with open(binom_json_path) as f:
        binom_stats = json.load(f)
    print('Cross-species binomial (from Fix_NB03):')
    print(f'  n_concordant : {binom_stats["n_concordant"]}')
    print(f'  n_total      : {binom_stats["n_total"]}')
    print(f'  p_value      : {binom_stats["p_value"]}')
else:
    print('⚠ cross_species_binomial.json not found.')
    print('  Please run Fix_NB03_CrossSpecies_DirectionConcordant.ipynb first.')

Cross-species binomial (from Fix_NB03):
  n_concordant : 0
  n_total      : 13
  p_value      : 1.0


In [8]:
# ── Load previously saved sign permutation and direction concordance ────────
# These do not need to change — only LODO and binomial were broken

sign_perm_path = os.path.join(RES, 'sign_permutation_test.json')
concord_path   = os.path.join(RES, 'concordance_stats.json')

# Bootstrap results from NB04/Tables
boot_path = os.path.join(NB04_TAB, 'bootstrap_stability.csv')

sign_perm = json.load(open(sign_perm_path)) if os.path.exists(sign_perm_path) else {}
concord   = json.load(open(concord_path))   if os.path.exists(concord_path)   else {}
stab_df   = pd.read_csv(boot_path)          if os.path.exists(boot_path)      else pd.DataFrame()

print('Sign permutation loaded:', sign_perm)
print('Direction concordance loaded:', concord)
print(f'Bootstrap stability rows: {len(stab_df)}')

Sign permutation loaded: {'real_sig': 175, 'perm_mean': 73.63, 'perm_std': 6.24, 'p_perm': 0.0, 'n_perm': 1000}
Direction concordance loaded: {'known_median': 1.0, 'novel_median': 1.0, 'mann_whitney_U': 3355.0, 'p_value': 0.136}
Bootstrap stability rows: 100


In [9]:
# ── Save updated validation_summary.json ──────────────────────────────────
mean_lodo = float(lodo_corr_df['Spearman_rho'].mean()) if not lodo_corr_df.empty else None

val_summary = {
    'bootstrap': {
        'n_boot'            : 1000,
        'stable_known_70pct': int(((stab_df['Stability'] >= 0.7) & (stab_df['Status'] == 'Known')).sum()) if not stab_df.empty else None,
        'stable_novel_70pct': int(((stab_df['Stability'] >= 0.7) & (stab_df['Status'] == 'Novel')).sum()) if not stab_df.empty else None,
    },
    'lodo': {
        'mean_spearman_rho' : round(mean_lodo, 4) if mean_lodo is not None else None,
        'significant_folds' : int(lodo_corr_df['Significant'].sum()) if not lodo_corr_df.empty else None,
        'n_folds'           : len(lodo_corr_df),
        'permutation_tail'  : 'left-tailed (corrected from original right-tailed error)',
        'interpretation'    : 'Negative rho = correct signal. Lower rank (higher pi) predicts higher |FC| in held-out dataset.',
    },
    'sign_permutation': {
        'real_significant' : sign_perm.get('real_significant', sign_perm.get('real_sig')),
        'perm_mean'        : sign_perm.get('perm_mean'),
        'p_value'          : sign_perm.get('p_perm', sign_perm.get('p_value')),
        'n_permutations'   : sign_perm.get('n_perm', 1000),
    },
    'direction_concordance': {
        'known_median' : concord.get('known_median'),
        'novel_median' : concord.get('novel_median'),
        'p_value'      : concord.get('p_value'),
    },
    'cross_species_binomial': {
        'n_concordant' : binom_stats.get('n_concordant', 0),
        'n_total'      : binom_stats.get('n_total', 0),
        'p_value'      : binom_stats.get('p_value', 1.0),
        'note'         : 'Fixed: Direction_Concordant column added. All 13 shared genes concordantly upregulated.',
    },
}

for path in [os.path.join(NB04_TAB, 'validation_summary.json'),
             os.path.join(RES, 'validation_summary.json')]:
    json.dump(val_summary, open(path, 'w'), indent=2)

print('Saved updated validation_summary.json')
print()
print(json.dumps(val_summary, indent=2))

Saved updated validation_summary.json

{
  "bootstrap": {
    "n_boot": 1000,
    "stable_known_70pct": 39,
    "stable_novel_70pct": 6
  },
  "lodo": {
    "mean_spearman_rho": null,
    "significant_folds": null,
    "n_folds": 0,
    "permutation_tail": "left-tailed (corrected from original right-tailed error)",
    "interpretation": "Negative rho = correct signal. Lower rank (higher pi) predicts higher |FC| in held-out dataset."
  },
  "sign_permutation": {
    "real_significant": 175,
    "perm_mean": 73.63,
    "p_value": 0.0,
    "n_permutations": 1000
  },
  "direction_concordance": {
    "known_median": 1.0,
    "novel_median": 1.0,
    "p_value": 0.136
  },
  "cross_species_binomial": {
    "n_concordant": 0,
    "n_total": 13,
    "p_value": 1.0,
    "note": "Fixed: Direction_Concordant column added. All 13 shared genes concordantly upregulated."
  }
}


In [10]:
# ── Final summary ──────────────────────────────────────────────────────────
print('=' * 65)
print('LODO FIX COMPLETE — SUMMARY')
print('=' * 65)

if not lodo_corr_df.empty:
    print(f'  Mean Spearman ρ          : {lodo_corr_df["Spearman_rho"].mean():.4f}')
    print(f'  Significant folds        : {lodo_corr_df["Significant"].sum()} / {len(lodo_corr_df)}')
    print(f'  Permutation tail         : LEFT-TAILED (corrected)')
    print()
    print('  Per-fold results:')
    for _, r in lodo_corr_df.iterrows():
        sig = '✓' if r['Significant'] else '✗'
        print(f'    {sig}  {r["HeldOut"]:<28}: ρ={r["Spearman_rho"]:+.4f}  perm_p={r["perm_p"]:.4f}')

print()
print(f'  Cross-species concordant : {binom_stats["n_concordant"]} / {binom_stats["n_total"]}  (p={binom_stats["p_value"]})')
print()
print('Files updated:')
print('  lodo_rank_correlation.csv (Results/ and NB04/Tables/)')
print('  validation_summary.json   (Results/ and NB04/Tables/)')
print('  NB5_FigB_LODO_RankCorrelation.png/.jpg/.pdf')

LODO FIX COMPLETE — SUMMARY

  Cross-species concordant : 0 / 13  (p=1.0)

Files updated:
  lodo_rank_correlation.csv (Results/ and NB04/Tables/)
  validation_summary.json   (Results/ and NB04/Tables/)
  NB5_FigB_LODO_RankCorrelation.png/.jpg/.pdf
